# Exploring the GBM drug-response tables

This notebook reads the committed tables (`data/processed`, `data/smiles`) and the latest pipeline results (`results/`). It performs no analysis of its own; see `docs/METHODS.md` for how the tables were built.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from gbm_drug import config as cfg
from gbm_drug.data_processing import load_drug_summary, load_gbm_dose_response

curves = load_gbm_dose_response()
summary = load_drug_summary()
print(f"{len(curves)} curves, {curves.cell_line.nunique()} GBM cell lines, {summary.shape[0]} drugs")
summary.head()

## GBM-selective sensitivity

`z_mean` is the mean GDSC z-score over GBM lines; negative means GBM lines are more sensitive than the pan-cancer panel. `z_q` is the BH-adjusted p of a one-sample t-test against 0.

In [ ]:
sel = summary[summary.gbm_selective]
print(len(sel), 'GBM-selective drugs')
sel[['drug_name','putative_target','pathway_name','n_cell_lines','z_mean','z_q','ic50_um_geomean']].sort_values('z_mean')

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
q = summary.z_q.clip(lower=1e-300)
ax.scatter(summary.z_mean, -np.log10(q), s=10, c=np.where(summary.gbm_selective, 'crimson', 'grey'))
ax.axhline(-np.log10(0.05), ls='--', c='k', lw=1); ax.axvline(0, ls=':', c='k', lw=1)
ax.set_xlabel('mean GBM z-score'); ax.set_ylabel('-log10 FDR q'); ax.set_title('Selectivity volcano');

## How many IC50s are inside the tested concentration range?

In [ ]:
within = curves.groupby('drug_name').ic50_within_range.mean()
print(f"median share of in-range IC50s per drug: {within.median():.2f}")
within.hist(bins=30); plt.xlabel('fraction of curves with IC50 <= max tested conc'); plt.ylabel('drugs');

## Cell-line coverage per drug

In [ ]:
summary.n_cell_lines.value_counts().sort_index().plot(kind='bar', figsize=(9,3)); plt.xlabel('GBM cell lines screened'); plt.ylabel('drugs');

## Structures

In [ ]:
from gbm_drug.feature_extraction import load_smiles_table
sm = load_smiles_table()
print(f"{len(sm)} drugs with structures; sources: {sm.source.value_counts().to_dict()}")
print(open(cfg.SMILES_DATA_DIR / 'unresolved.txt').read().count('\n'), 'drugs without a structure (see data/smiles/unresolved.txt)')

## Latest benchmark results (if `python main.py` has been run)

In [ ]:
p = cfg.BENCHMARK_RESULTS_DIR / 'summary.csv'
if p.exists():
    s = pd.read_csv(p)
    view = s[(s.task=='gbm_selectivity') & (s.metric=='spearman_rho')].pivot(index='model', columns='strategy', values='mean').sort_values('grouped', ascending=False)
    display(view.round(3))
else:
    print('run `python main.py --quick` first')